# Train YOLO26 - Phát hiện biển số xe (BSD/BSV)

Dataset: 3433 ảnh train / 1145 ảnh val, 2 lớp: `BSD` (biển đen), `BSV` (biển vàng).

**Trước khi chạy:** `Runtime → Change runtime type → T4 GPU` (miễn phí).

⚠️ **QUAN TRỌNG — đừng dùng Colab "Add data" để mount dataset Kaggle.** Mount `/kaggle/input/...` là ổ mạng read-only,
đọc chậm (~30 MB/s) và không cache được → GPU nằm chờ data, train chậm gấp 3-5 lần.
Hãy chạy cell **Cách A** bên dưới: `kagglehub` tải dataset về đĩa local rồi copy ra `/content` (SSD local) + cache vào RAM.

In [ ]:
# 1. Kiểm tra GPU
import torch
assert torch.cuda.is_available(), "Không thấy GPU! Chọn Runtime -> Change runtime type -> T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

In [ ]:
# 2. Cài thư viện
!pip install -q ultralytics kagglehub
import ultralytics
ultralytics.checks()

## 3. Dataset — CÁCH A (tải từ Kaggle bằng kagglehub)

Chuẩn bị Kaggle API token (chỉ cần làm 1 lần):
1. Vào https://www.kaggle.com/settings → API → `Create New Token` (tải về `kaggle.json`).
2. Trong Colab: biểu tượng 🔑 (Secrets) ở thanh bên trái → thêm `KAGGLE_USERNAME` và `KAGGLE_KEY`.

In [ ]:
# CÁCH A: tải dataset từ Kaggle về đĩa local, rồi copy ra /content (SSD local của Colab)
import os
from google.colab import userdata

os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

import kagglehub
src = kagglehub.dataset_download("duydieunguyen/licenseplates")
print("Kagglehub tải về tại:", src)

# Copy ra /content để có đường dẫn cố định, đọc nhanh
!rm -rf /content/dataset
!cp -r "$src" /content/dataset
ds_path = "/content/dataset"
!ls "$ds_path"

In [ ]:
# (CHỈ chạy cell này nếu bạn lỡ mount dataset qua Colab "Add data" — copy từ mount chậm ra local SSD)
# !rm -rf /content/dataset
# !cp -r /kaggle/input/licenseplates /content/dataset
# ds_path = "/content/dataset"
# !ls "$ds_path"

In [ ]:
# 4. Xác định thư mục dataset và tạo data.yaml
from pathlib import Path

# Tự tìm thư mục chứa images/train bên trong dataset
root = None
for p in Path(ds_path).rglob("images/train"):
    if p.is_dir():
        root = p.parent.parent
        break
assert root is not None, f"Không tìm thấy images/train trong {ds_path} — kiểm tra lại cấu trúc dataset"
root = root.resolve()
print("Gốc dataset:", root)

n_train = len(list((root / "images/train").glob("*.*")))
n_val = len(list((root / "images/val").glob("*.*")))
print(f"Ảnh train: {n_train} | Ảnh val: {n_val}")

DATA_YAML = "/content/data.yaml"
with open(DATA_YAML, "w") as f:
    f.write(f"""train: {root}/images/train
val: {root}/images/val

nc: 2
names: ['BSD', 'BSV']
""")
print("Đã ghi", DATA_YAML)
!cat "$DATA_YAML"

In [ ]:
# 5. TRAIN — hyperparameters giống train.py trong repo
# Tốc độ thực tế trên T4 free (đã cache RAM): yolo26m ~1.4 it/s => ~2.5 phút/epoch.
# 100 epoch ~ 4 giờ; early stopping (patience=20) thường dừng ở epoch 40-70 => ~2-3 giờ.
# Muốn nhanh hơn: FAST=True (yolo26n + imgsz 512) => ~1-1.5 giờ, mAP giảm một chút.
from ultralytics import YOLO

FAST = False           # True: yolo26n + imgsz 512 — nhanh gấp ~3 lần, đánh đổi độ chính xác
MODEL = "yolo26n.pt" if FAST else "yolo26m.pt"
EPOCHS = 100
BATCH = 16             # T4 16GB (đang dùng ~8.7G); có thể thử 24-32, OOM thì giảm về 16
IMGSZ = 512 if FAST else 640

model = YOLO(MODEL)
results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=0,
    cache="ram",       # dataset ~1GB, Colab ~13GB RAM -> cache hết vào RAM, hết nghẽn I/O
    patience=20,
    save=True,
    save_period=10,
    project="runs/train",
    name="license_plate_yolo26",
    exist_ok=True,
    pretrained=True,
    optimizer="auto",
    lr0=0.01,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3,
    warmup_momentum=0.8,
    warmup_bias_lr=0.1,
    box=7.5,
    cls=0.5,
    dfl=1.5,
    nbs=64,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=0.0,
    translate=0.1,
    scale=0.5,
    shear=0.0,
    perspective=0.0,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.0,
    copy_paste=0.0,
    erasing=0.4,
)

In [ ]:
# 6. Đánh giá model tốt nhất trên tập val
from pathlib import Path
from ultralytics import YOLO

# Tự tìm best.pt (ultralytics bản mới có thể lồng đường dẫn dưới runs/detect/...)
hits = sorted(Path("/content").glob("runs/**/license_plate_yolo26/weights/best.pt"))
assert hits, "Chưa có best.pt — chạy cell train trước đã"
BEST = str(hits[-1])
print("Best model:", BEST)

best = YOLO(BEST)
metrics = best.val(data=DATA_YAML, imgsz=640, batch=16, device=0)
print(f"mAP50:    {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")

In [ ]:
# 7. Lưu weights ra Google Drive để tải về máy
import shutil
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")

dst = Path("/content/drive/MyDrive/rlvd_weights")
dst.mkdir(parents=True, exist_ok=True)
wdir = Path(BEST).parent
shutil.copy2(wdir / "best.pt", dst / "best.pt")
shutil.copy2(wdir / "last.pt", dst / "last.pt")
print("Đã lưu best.pt + last.pt vào", dst)

# Hoặc tải trực tiếp từ Colab:
# from google.colab import files
# files.download(BEST)

## Ghi chú
- Nếu Colab bị disconnect giữa chừng: chạy lại cell 1–4 rồi chạy cell 5 — Ultralytics sẽ **resume** từ `last.pt` nếu có.
- Nếu vẫn thấy `Slow image access` khi train: kiểm tra lại đã chạy cell Cách A (copy ra `/content/dataset`) chưa, data.yaml phải trỏ về `/content/dataset/...`.
- Sau khi tải `best.pt` về, copy vào `runs/train/license_plate_yolo26/weights/` của repo để dùng với `run_pipeline.py` / `infer.py`.